# VibeRec — Technical Write-Up

**Days 29–30:** Architecture · Training · Evaluation · Ablation · Introduction · Related Work · Conclusion

---

In [ ]:
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
import config as cfg

NOTEBOOKS_DIR = cfg.NOTEBOOKS_DIR
print('Project root:', PROJECT_ROOT)

---
## Part I — Introduction

### Problem Motivation

Collaborative filtering (CF) recommenders — the industry standard since Netflix Prize — predict what a user will like based on who else liked the same things. They are powerful but blind to *why* a movie appeals: a user who loved *Moonlight* because of its quiet, intimate pacing will not be well-served by a CF system that also liked *Black Panther* simply because both were popular in a demographic cohort.

Content-based systems address this by encoding movie attributes, but shallow attribute matching (genre, director) misses the gestalt — the *vibe* — of a film: its emotional register, narrative rhythm, tonal texture.

**VibeRec** learns a 128-dimensional *vibe embedding* for each movie from raw plot summaries, user-generated tags, and structured metadata using a CNN+LSTM encoder trained with triplet loss. These embeddings are blended with NCF collaborative scores via a tuneable α parameter, giving users explicit control over the exploration–relevance trade-off.

### Contributions
1. A CNN+LSTM vibe encoder trained end-to-end on MovieLens 25M + CMU plot summaries.
2. A hybrid scoring function with a single interpretable knob (α) that interpolates between content vibe and collaborative novelty signals.
3. Comprehensive evaluation: VibeRec vs. three baselines + ablation study across five component variants.
4. A Streamlit demo with UMAP vibe-space visualisation.

---
## Part II — Architecture

The system has four stages:
1. **Text encoding** — plot summaries (CNN) and tag sequences (LSTM) are encoded into 128-d vectors.
2. **Metadata projection** — 23-dim one-hot genre + normalised scalars → 64-d.
3. **Vibe embedding** — concatenation → MLP → 128-d unit-norm vibe vector.
4. **Recommendation** — α-weighted blend of vibe cosine similarity and NCF collaborative score.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

COLORS = {
    'input':   '#FFF9C4',
    'encoder': '#BBDEFB',
    'combine': '#C8E6C9',
    'score':   '#FFCCBC',
    'output':  '#E1BEE7',
}

def box(ax, x, y, w, h, title, sub='', color='white', fontsize=9):
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                          facecolor=color, edgecolor='#555', linewidth=1.2, zorder=2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2 + (0.1 if sub else 0), title,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', zorder=3)
    if sub:
        ax.text(x + w/2, y + h/2 - 0.18, sub,
                ha='center', va='center', fontsize=7, color='#444', zorder=3)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5), zorder=1)

# ── Input ──
box(ax, 0.2, 4.9, 2.4, 0.8, 'Plot Summary', '500 tokens (GloVe 300d)', COLORS['input'])
box(ax, 0.2, 3.6, 2.4, 0.8, 'Tag Sequences', '200 tokens (GloVe 300d)', COLORS['input'])
box(ax, 0.2, 2.3, 2.4, 0.8, 'Metadata', '23-dim (genre + scalars)', COLORS['input'])

# ── Encoders ──
box(ax, 3.2, 4.9, 2.8, 0.8, 'CNN Encoder', 'filters 3,4,5 × 128 → proj 128d', COLORS['encoder'])
box(ax, 3.2, 3.6, 2.8, 0.8, 'LSTM Encoder', '2-layer bidirectional → 128d', COLORS['encoder'])
box(ax, 3.2, 2.3, 2.8, 0.8, 'Metadata MLP', 'Linear(23→64) + ReLU', COLORS['encoder'])

# ── Concat & Vibe MLP ──
box(ax, 6.8, 3.6, 1.6, 0.8, 'Concat', '128+128+64 = 320d', COLORS['combine'])
box(ax, 9.1, 3.6, 2.2, 0.8, 'Vibe MLP', 'Linear(320→128)\nL2 normalise', COLORS['combine'])

# ── Scoring ──
box(ax, 6.8, 1.4, 2.2, 0.8, 'FAISS ANN', 'cosine sim (α weight)', COLORS['score'])
box(ax, 9.1, 1.4, 2.2, 0.8, 'NCF MLP', '256→128→64→1  sigmoid', COLORS['score'])

# ── Novelty & blend ──
box(ax, 12.0, 3.6, 1.8, 0.8, 'Movie\nVibe DB', '12k × 128d FAISS', COLORS['combine'])
box(ax, 12.0, 1.4, 1.8, 0.8, 'Novelty\nScore', 'popularity + density', COLORS['score'])
box(ax, 12.0, 5.4, 1.8, 0.8, 'α-Blend', 'final score', COLORS['output'])

# ── Arrows: inputs → encoders ──
for y in [5.3, 4.0, 2.7]:
    arrow(ax, 2.6, y, 3.2, y)

# ── Encoders → concat ──
arrow(ax, 6.0, 5.3, 6.8, 4.2)
arrow(ax, 6.0, 4.0, 6.8, 4.0)
arrow(ax, 6.0, 2.7, 6.8, 3.8)

# ── Concat → Vibe MLP ──
arrow(ax, 8.4, 4.0, 9.1, 4.0)

# ── Vibe MLP → FAISS ──
arrow(ax, 10.2, 3.6, 10.2, 2.2)
arrow(ax, 9.9, 3.6, 7.9, 2.2)

# ── Vibe MLP → Movie DB ──
arrow(ax, 11.3, 4.0, 12.0, 4.0)

# ── FAISS + NCF → alpha blend ──
arrow(ax, 9.0, 1.8, 12.4, 5.4)
arrow(ax, 11.3, 1.8, 12.4, 5.4)

# ── Novelty → alpha blend ──
arrow(ax, 12.9, 2.2, 12.9, 5.4)

# ── Alpha label ──
ax.text(11.5, 4.9, 'α', fontsize=14, color='#d32f2f', fontweight='bold')
ax.text(11.5, 4.6, '1-α', fontsize=11, color='#1565c0', fontweight='bold')

ax.set_title('VibeRec Architecture', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
arch_path = os.path.join(NOTEBOOKS_DIR, 'architecture_diagram.png')
plt.savefig(arch_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Saved {arch_path}')

---
## Part III — Dataset Statistics

In [ ]:
import pickle

movies  = pd.read_csv(cfg.MOVIES_MASTER)
train_r = pd.read_csv(cfg.RATINGS_TRAIN, usecols=['userId','movieId','rating'])
test_r  = pd.read_csv(cfg.RATINGS_TEST,  usecols=['userId','movieId','rating'])

with open(cfg.MOVIE_ID_TO_IDX, 'rb') as f:
    m2i = pickle.load(f)

n_movies_total = len(movies)
n_movies_emb   = len(m2i)
n_users_train  = train_r['userId'].nunique()
n_users_test   = test_r['userId'].nunique()
n_train        = len(train_r)
n_test         = len(test_r)
density_train  = n_train / (n_users_train * n_movies_emb) * 100
density_test   = n_test  / (n_users_test  * n_movies_emb) * 100

stats = pd.DataFrame([
    {'Split': 'Train (80%)',
     'Movies': f'{n_movies_emb:,}',
     'Users':  f'{n_users_train:,}',
     'Ratings': f'{n_train:,}',
     'Density': f'{density_train:.4f}%'},
    {'Split': 'Test (20%)',
     'Movies': f'{n_movies_emb:,}',
     'Users':  f'{n_users_test:,}',
     'Ratings': f'{n_test:,}',
     'Density': f'{density_test:.4f}%'},
    {'Split': 'Total',
     'Movies': f'{n_movies_total:,} raw / {n_movies_emb:,} with embeddings',
     'Users':  f'{max(n_users_train, n_users_test):,}',
     'Ratings': f'{n_train+n_test:,}',
     'Density': '—'},
])

print('Dataset Statistics')
print('=' * 70)
print(stats.to_string(index=False))

print(f'\nRating scale: 0.5–5.0  |  High-rating threshold ("liked"): ≥ {cfg.HIGH_RATING_THRESH}')
print(f'Min ratings per movie to include: {cfg.MIN_RATINGS}')
print(f'Text sources: CMU plot summaries + MovieLens tag genome')
print(f'Word embeddings: GloVe 6B 300d  |  Vocab size: {cfg.VOCAB_SIZE:,}')

---
## Part IV — Training Details

In [ ]:
training_details = pd.DataFrame([
    {
        'Component':   'VibeEncoder (CNN+LSTM)',
        'Epochs':      cfg.VIBE_EPOCHS,
        'Batch size':  cfg.VIBE_BATCH_SIZE,
        'Loss':        f'Triplet (margin={cfg.TRIPLET_MARGIN})',
        'Optimizer':   'Adam',
        'LR':          cfg.VIBE_LR,
        'Weight decay': cfg.VIBE_WEIGHT_DECAY,
        'Dropout':     cfg.DROPOUT_VIBE,
    },
    {
        'Component':   'NCF (MLP)',
        'Epochs':      cfg.NCF_EPOCHS,
        'Batch size':  cfg.NCF_BATCH_SIZE,
        'Loss':        'BCE (with neg sampling)',
        'Optimizer':   'Adam',
        'LR':          cfg.NCF_LR,
        'Weight decay': '—',
        'Dropout':     f'{cfg.DROPOUT_NCF_1} / {cfg.DROPOUT_NCF_2}',
    },
])

arch_details = pd.DataFrame([
    {'Module': 'CNN',    'Config': f'{len(cfg.CNN_KERNELS)} kernel sizes {cfg.CNN_KERNELS}, {cfg.CNN_FILTERS} filters each → project to {cfg.CNN_OUT_DIM}d'},
    {'Module': 'LSTM',   'Config': f'{cfg.LSTM_LAYERS}-layer, hidden={cfg.LSTM_HIDDEN}, proj to {cfg.LSTM_OUT_DIM}d'},
    {'Module': 'Meta',   'Config': f'Linear({cfg.METADATA_DIM} → {cfg.METADATA_PROJ_DIM}) + ReLU'},
    {'Module': 'Concat', 'Config': f'{cfg.CNN_OUT_DIM} + {cfg.LSTM_OUT_DIM} + {cfg.METADATA_PROJ_DIM} = {cfg.CNN_OUT_DIM+cfg.LSTM_OUT_DIM+cfg.METADATA_PROJ_DIM}d'},
    {'Module': 'VibeOut','Config': f'Linear → {cfg.VIBE_EMBED_DIM}d, L2 normalised'},
    {'Module': 'NCF MLP','Config': f'Input={cfg.VIBE_EMBED_DIM*2}d → {" → ".join(str(h) for h in cfg.NCF_HIDDEN)} → 1 (sigmoid)'},
])

print('Training Hyperparameters')
print('=' * 80)
print(training_details.to_string(index=False))

print('\nArchitecture Details')
print('=' * 80)
print(arch_details.to_string(index=False))

print(f'\nNCF negative sampling ratio: {cfg.NCF_NEG_RATIO}x  |  Eval K: {cfg.EVAL_K}')

### Training Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

vibe_curve = os.path.join(cfg.MODELS_DIR, 'vibe_encoder_loss_curve.png')
ncf_curve  = os.path.join(cfg.MODELS_DIR, 'day19_ncf_training.png')

for ax, path, title in [
    (axes[0], vibe_curve, 'VibeEncoder — Triplet Loss'),
    (axes[1], ncf_curve,  'NCF — BCE Loss / AUC / HR@10'),
]:
    if os.path.exists(path):
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'Not found:\n{path}', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.show()

---
## Part V — Evaluation Results

In [ ]:
vibrec_df    = pd.read_csv(os.path.join(NOTEBOOKS_DIR, 'results_vibrec.csv'))
baselines_df = pd.read_csv(os.path.join(NOTEBOOKS_DIR, 'results_baselines.csv'))

# Aggregate VibeRec per-user results
vibrec_agg = vibrec_df[['precision','recall','ndcg','hit_rate','mrr']].agg(['mean','std'])

# Pivot baselines: model × metric → mean / std
bl_pivot = baselines_df.pivot_table(index='model', columns='metric',
                                    values=['mean','std'], aggfunc='first')

K = cfg.EVAL_K
metrics = ['precision', 'recall', 'ndcg', 'hit_rate', 'mrr']
metric_labels = {
    'precision': f'P@{K}', 'recall': f'R@{K}',
    'ndcg': f'NDCG@{K}', 'hit_rate': f'HR@{K}', 'mrr': 'MRR'
}

rows = []
for model in ['Popularity', 'Genre-Match', 'SVD(k=64)', 'VibeRec']:
    row = {'Model': model}
    if model == 'VibeRec':
        for m in metrics:
            mu  = vibrec_agg.loc['mean', m]
            std = vibrec_agg.loc['std',  m]
            row[metric_labels[m]] = f'{mu:.4f} ± {std:.4f}'
    else:
        sub = baselines_df[baselines_df['model'] == model]
        for m in metrics:
            r = sub[sub['metric'] == m]
            if r.empty:
                row[metric_labels[m]] = '—'
            else:
                row[metric_labels[m]] = f"{r['mean'].values[0]:.4f} ± {r['std'].values[0]:.4f}"
    rows.append(row)

results_table = pd.DataFrame(rows).set_index('Model')
print(f'Evaluation Results (K={K}, 1000 test users)')
print('=' * 100)
print(results_table.to_string())

print('\nNote: VibeRec evaluated at α=0.8 (default). SVD is the strongest baseline.')

In [ ]:
# Bar chart comparison
models = ['Popularity', 'Genre-Match', 'SVD(k=64)', 'VibeRec']
metric_vals = {}
for m in ['precision', 'recall', 'ndcg', 'hit_rate', 'mrr']:
    vals = []
    for model in models:
        if model == 'VibeRec':
            vals.append(vibrec_agg.loc['mean', m])
        else:
            sub = baselines_df[(baselines_df['model'] == model) & (baselines_df['metric'] == m)]
            vals.append(sub['mean'].values[0] if not sub.empty else 0)
    metric_vals[m] = vals

x = np.arange(len(models))
width = 0.15
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#4CAF50','#2196F3','#FF9800','#9C27B0','#F44336']
for i, (m, label) in enumerate(metric_labels.items()):
    ax.bar(x + i*width, metric_vals[m], width, label=label, color=colors[i], alpha=0.85)

ax.set_xticks(x + 2*width)
ax.set_xticklabels(models, fontsize=10)
ax.set_ylabel('Score')
ax.set_title(f'Model Comparison @ K={K}', fontsize=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
comp_path = os.path.join(NOTEBOOKS_DIR, 'model_comparison.png')
plt.savefig(comp_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Saved {comp_path}')

---
## Part VI — Ablation Study

In [ ]:
ablation_df = pd.read_csv(os.path.join(NOTEBOOKS_DIR, 'results_ablation.csv'))
print('Ablation Study — Component Contributions')
print('=' * 75)
print(ablation_df.to_string(index=False))

# Bar chart
metrics_abl  = ['precision', 'recall', 'ndcg', 'hit_rate', 'mrr']
variant_labels = {
    'A: CNN-only':   'A: CNN only',
    'B: LSTM-only':  'B: LSTM only',
    'C: No metadata':'C: No metadata',
    'D: No NCF':     'D: No NCF',
    'E: No novelty': 'E: No novelty',
    'Full VibeRec':  'Full VibeRec',
}

x = np.arange(len(ablation_df))
fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
for i, m in enumerate(metrics_abl):
    ax = axes[i]
    bars = ax.bar(x, ablation_df[m],
                  color=['#ef9a9a']*5 + ['#66bb6a'],
                  edgecolor='#555', linewidth=0.8)
    ax.set_title(metric_labels[m], fontsize=10, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['A','B','C','D','E','Full'], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, ablation_df[m]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

fig.suptitle('Ablation Study — Removing One Component at a Time',
             fontsize=12, fontweight='bold')
plt.tight_layout()
abl_path = os.path.join(NOTEBOOKS_DIR, 'ablation_chart.png')
plt.savefig(abl_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'[OK] Saved {abl_path}')

### Ablation Analysis

| Variant | Precision@10 | vs Full | Interpretation |
|---------|------------|---------|----------------|
| A: CNN only | 0.0048 | −80% | LSTM tag context is dominant |
| B: LSTM only | 0.0104 | −56% | CNN plot signal adds meaningful diversity |
| C: No metadata | 0.0040 | −83% | Genre/runtime features are critical |
| D: No NCF | 0.0068 | −71% | Collaborative signal essential for relevance |
| E: No novelty | 0.0232 | −2% | Novelty has marginal ranking impact |
| **Full VibeRec** | **0.0238** | — | All components contribute |

**Key finding:** The metadata features (Variant C) contribute most strongly — removing them causes an 83% drop. The novelty score (Variant E) has the smallest individual impact, suggesting its value is more qualitative (surfacing lesser-known films) than precision-metric-measurable.

---
## Part VII — UMAP Visualisations

In [ ]:
umap_plots = [
    ('day25_umap_genre.png',     'UMAP — coloured by primary genre'),
    ('day25_umap_rating.png',    'UMAP — coloured by average IMDb rating'),
    ('day25_umap_annotated.png', 'UMAP — annotated with landmark movies'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (fname, title) in zip(axes, umap_plots):
    path = os.path.join(NOTEBOOKS_DIR, fname)
    if os.path.exists(path):
        ax.imshow(plt.imread(path))
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'Missing: {fname}', ha='center', va='center',
                transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.show()
print('[OK] UMAP panels displayed')

---
## Part VIII — Related Work

### Collaborative Filtering
**He et al. (2017)** — *Neural Collaborative Filtering* (NCF) — replaced matrix factorisation dot-product with a multi-layer perceptron, demonstrating that non-linear interaction modelling outperforms SVD on implicit feedback data. VibeRec uses NCF as its collaborative component, replacing the traditional user/item ID embeddings with pre-computed vibe taste profiles.

### Text-Based Content Encoding
**Kim (2014)** — *Convolutional Neural Networks for Sentence Classification* — showed that 1D CNNs with multiple kernel sizes capture local n-gram features effectively from fixed pre-trained word embeddings. VibeRec applies this architecture to movie plot summaries.

**Hochreiter & Schmidhuber (1997)** — *Long Short-Term Memory* — LSTMs model sequential dependencies in text, complementing the CNN's local feature extraction with global sequence understanding. VibeRec uses a 2-layer LSTM on tag sequences.

### Content-Based & Hybrid Recommenders
**Pazzani & Billsus (2007)** — *Content-Based Recommendation Systems* — reviews the family of systems that represent item features explicitly. VibeRec extends this line by learning latent vibe representations rather than relying on handcrafted attributes.

**Burke (2002)** — *Hybrid Recommender Systems: Survey and Experiments* — establishes the taxonomy of hybrid systems (weighted, switching, cascade). VibeRec implements a **weighted hybrid** via the α parameter, but with an interpretable user-facing knob rather than a fixed blend.

---
## Part IX — Conclusion

### What Worked
- The CNN+LSTM vibe encoder successfully captures tonal and narrative similarity: comedies cluster together, slow-burn dramas form a distinct region, and action/adventure fills another. The UMAP visualisations confirm non-trivial structure in the learned space.
- The α knob gives users a transparent, controllable exploration–exploitation lever. At α=1.0 recommendations are stylistically coherent; at α=0.5 they balance coherence with collaborative evidence.
- Metadata features (genre one-hot + rating/runtime scalars) turned out to be the single most impactful component (−83% precision when removed), validating the multi-modal fusion design.
- The full system beats Popularity and Genre-Match baselines across all metrics and competes with SVD on NDCG and HR@10.

### What Didn't Work
- **Cold-start:** new users with no ratings have no taste profile. The system needs at least a few ratings before it can personalise.
- **Review/tag noise:** MovieLens tags are user-generated and often noisy (genre labels, actor names). A cleaner review source (Letterboxd) would improve the LSTM's vibe signal.
- **SVD outperforms on Precision@10:** pure collaborative filtering (SVD k=64) still beats VibeRec on precision, suggesting the vibe signal trades precision for diversity/novelty.
- **α=0.0 is seed-blind:** pure novelty scoring is independent of the seed, limiting its practical use. A better formulation would be novelty conditioned on the seed's neighbourhood.

### Future Work
1. **Transformer encoder:** Replace CNN+LSTM with a Sentence-BERT or fine-tuned BERT encoder for richer plot representations.
2. **Multi-modal with poster images:** A ResNet-based visual encoder could capture visual aesthetic (colour palette, cinematography style) as an additional vibe dimension.
3. **Real reviews from Letterboxd:** User-written reviews carry richer tonal signal than tag sequences. Scraping or using a Letterboxd dataset would significantly improve the LSTM's input quality.
4. **Online α personalisation:** Learn a per-user optimal α from implicit feedback (click-through, watch completion) rather than relying on a global default.
5. **Contrastive pre-training:** Pre-train the vibe encoder with contrastive loss on (movie, positive review, negative review) triples before fine-tuning on triplet movie similarity.